# Karatsuba Length Generalization
**Goal:** Train on 4-bit + 8-bit, generalize to 16-bit (never seen).

**Architecture:** Looped transformer + NoPE + input injection + BFS traces

**Runtime:** A100 GPU recommended. Runtime → Change runtime type → A100.

In [ ]:
# Cell 1: Setup
import os
if not os.path.exists('/content/recursive-transformers/src'):
    !git clone https://github.com/dhruvsyam123/recursive-transformers.git /content/recursive-transformers
else:
    !cd /content/recursive-transformers && git pull
%cd /content/recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops

import jax
print(f'JAX devices: {jax.devices()}')
print(f'JAX backend: {jax.default_backend()}')

In [ ]:
# Cell 2: Model — NoPE + Input Injection + Sinusoidal Timestep
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import math

from src.model.looped_transformer import (
    SharedTransformerLayers, RMSNorm, MultiHeadSelfAttention, FeedForward,
    count_parameters,
)


class LoopedTransformerBlock(eqx.Module):
    """Transformer block with SINUSOIDAL timestep (not learned)."""
    attention: MultiHeadSelfAttention
    ffn: FeedForward
    norm1: RMSNorm
    norm2: RMSNorm
    d_model: int = eqx.field(static=True)

    def __init__(self, d_model, n_heads, d_ff, *, key):
        k1, k2 = jax.random.split(key)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, key=k1)
        self.ffn = FeedForward(d_model, d_ff, key=k2)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.d_model = d_model

    def sinusoidal_timestep(self, t):
        """Fixed sinusoidal timestep encoding — generalizes to any t."""
        d = self.d_model
        i = jnp.arange(0, d, 2, dtype=jnp.float32)
        freq = jnp.exp(-i * (math.log(10000.0) / d))
        angle = t * freq
        emb = jnp.zeros(d)
        emb = emb.at[0::2].set(jnp.sin(angle))
        emb = emb.at[1::2].set(jnp.cos(angle))
        return emb

    def __call__(self, x, timestep, mask=None):
        t_emb = self.sinusoidal_timestep(timestep)
        x_cond = x + t_emb[None, :]
        normed = jax.vmap(self.norm1)(x_cond)
        x = x + self.attention(normed, mask=mask)
        normed = jax.vmap(self.norm2)(x)
        x = x + jax.vmap(self.ffn)(normed)
        return x


class KaratsubaModel(eqx.Module):
    """Looped transformer with NoPE + input injection.

    - No positional encoding (NoPE): causal mask + tag tokens provide ordering
    - Input injection: original embedding added back at each loop iteration
    - Sinusoidal timestep: generalizes to unseen loop counts
    """
    token_embed: eqx.nn.Embedding
    block: LoopedTransformerBlock
    final_norm: RMSNorm
    output_head: eqx.nn.Linear
    inject_scale: jnp.ndarray  # learnable scale for input injection

    def __call__(self, tokens, n_loops):
        # 1. Embed tokens (NO position encoding)
        x0 = jax.vmap(self.token_embed)(tokens)
        x = x0

        # 2. Causal mask
        seq_len = tokens.shape[0]
        mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))

        # 3. Looped transformer with input injection
        def scan_fn(hidden, timestep):
            # Input injection: add original embedding back (scaled)
            hidden = hidden + self.inject_scale * x0
            hidden = self.block(hidden, timestep, mask=mask)
            return hidden, None

        x, _ = jax.lax.scan(scan_fn, x, jnp.arange(n_loops))

        # 4. Output
        x = jax.vmap(self.final_norm)(x)
        return jax.vmap(self.output_head)(x)


D_MODEL = 256
VOCAB = 143
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)

model = KaratsubaModel(
    token_embed=eqx.nn.Embedding(VOCAB, D_MODEL, key=k1),
    block=LoopedTransformerBlock(D_MODEL, n_heads=8, d_ff=512, key=k2),
    final_norm=RMSNorm(D_MODEL),
    output_head=eqx.nn.Linear(D_MODEL, VOCAB, key=k3),
    inject_scale=jnp.array(0.1),  # start small, learnable
)
print(f'Parameters: {count_parameters(model):,}')

In [ ]:
# Cell 3: Datasets (BFS ordering) + Loss Mask
from src.data.dataset import MultiplicationDataset, DataConfig
from src.data.tokenizer import Tokenizer

tok = Tokenizer()
INPUT_ID = tok.encode_token('[INPUT]')
BASE_MUL_ID = tok.encode_token('[BASE_MUL]')
SPLIT_ID = tok.encode_token('[SPLIT]')

# 4-bit (base case) — depth-first is fine for base cases (no recursion)
ds4 = MultiplicationDataset(DataConfig(
    bit_widths=[4], algorithm='karatsuba', base_case_bits=4,
    train_fraction=1.0, seed=42,
))

# 8-bit (1 recursion level) — BREADTH-FIRST ordering
ds8 = MultiplicationDataset(DataConfig(
    bit_widths=[8], algorithm='karatsuba', base_case_bits=4,
    ordering='breadth_first',  # KEY CHANGE: BFS
    max_samples_per_width=4000, seed=42,
))

ml4 = ds4.get_max_seq_len('train') + 2
ml8 = ds8.get_max_seq_len('train') + 2
print(f'4-bit: {ds4.train_size()} examples, max_len={ml4}')
print(f'8-bit: {ds8.train_size()} train / {ds8.test_size()} test, max_len={ml8}')

# Show a sample BFS trace
from src.data.karatsuba_trace import KaratsubaTraceGenerator
gen = KaratsubaTraceGenerator(base_case_bits=4)
trace = gen.generate(179, 214, 8, ordering='breadth_first')
print(f'\nSample BFS trace ({len(trace.steps)} steps):')
print(gen.trace_to_string(trace, show_description=True)[:500])
print('...')

# Loss mask: mask out top-level input bits
def make_mask(token_ids_full, pad_mask_full):
    BIT_0, BIT_1 = tok.encode_token(0), tok.encode_token(1)
    is_bit = (token_ids_full == BIT_0) | (token_ids_full == BIT_1)
    seen = jnp.cumsum(
        (token_ids_full == SPLIT_ID) | (token_ids_full == BASE_MUL_ID), axis=1
    ) > 0
    predictable = ~(~seen & is_bit)
    return (pad_mask_full * predictable.astype(jnp.float32))[:, 1:]

In [ ]:
# Cell 4: Training — Grokfast + weight decay 0.15 + randomized loops
import random

# --- Grokfast (EMA gradient filter for faster grokking) ---
# From: ironjr/grokfast — amplifies slow gradient components
class GrokfastEMA:
    def __init__(self, alpha=0.98, lamb=2.0):
        self.alpha = alpha
        self.lamb = lamb
        self.ema = None

    def __call__(self, grads):
        leaves, treedef = jax.tree.flatten(grads)
        if self.ema is None:
            self.ema = [jnp.zeros_like(g) for g in leaves]
        new_ema = []
        new_grads = []
        for g, e in zip(leaves, self.ema):
            e_new = self.alpha * e + (1 - self.alpha) * g
            new_ema.append(e_new)
            new_grads.append(g + self.lamb * e_new)
        self.ema = new_ema
        return treedef.unflatten(new_grads)

grokfast = GrokfastEMA(alpha=0.98, lamb=2.0)

# --- Optimizer ---
schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=1e-3, warmup_steps=500,
    decay_steps=50000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.15)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

# --- Training step ---
def make_train_step(n_loops):
    @eqx.filter_jit
    def train_step(model, opt_state, token_ids_full, pad_mask_full):
        input_ids = token_ids_full[:, :-1]
        target_ids = token_ids_full[:, 1:]
        mask = make_mask(token_ids_full, pad_mask_full)

        def loss_fn(model):
            def fwd(ids):
                return model(ids, n_loops)
            logits = jax.vmap(fwd)(input_ids)
            log_probs = jax.nn.log_softmax(logits, axis=-1)
            targets_oh = jax.nn.one_hot(target_ids, logits.shape[-1])
            tok_loss = -jnp.sum(targets_oh * log_probs, axis=-1)
            return jnp.sum(tok_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)

        loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
        return loss, grads
    return train_step

# Pre-compile for different loop counts
train_steps = {n: make_train_step(n) for n in [4, 6, 8, 10, 12]}

# --- Eval ---
def quick_eval(model, n_bits, n_loops, n_samples=200, ordering='breadth_first'):
    ev = MultiplicationDataset(DataConfig(
        bit_widths=[n_bits], algorithm='karatsuba', base_case_bits=4,
        ordering=ordering,
        max_samples_per_width=n_samples, train_fraction=0.0, seed=999,
    ))
    ml = ev.get_max_seq_len('test') + 2
    correct, total = 0, 0
    for batch in ev.get_batch(split='test', batch_size=16, shuffle=False, max_len=ml):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        inp = tk[:, :-1]
        tgt = tk[:, 1:]
        em = make_mask(tk, pm)
        def fwd(ids):
            return model(ids, n_loops)
        logits = jax.vmap(fwd)(inp)
        preds = jnp.argmax(logits, axis=-1)
        matches = (preds == tgt) | (em == 0)
        correct += int(jnp.all(matches, axis=-1).sum())
        total += len(batch['x'])
    return correct, total

# --- Training loop ---
rng = random.Random(42)

print('Training (NoPE + input injection + BFS + Grokfast)...')
for epoch in range(500):
    epoch_loss, n_batches = 0.0, 0

    # 4-bit batches
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in ds4.get_batch(split='train', batch_size=64, max_len=ml4):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        loss, grads = step_fn(model, opt_state, tk, pm)
        grads = grokfast(grads)
        updates, opt_state = optimizer.update(
            grads, opt_state, eqx.filter(model, eqx.is_array)
        )
        model = eqx.apply_updates(model, updates)
        epoch_loss += float(loss)
        n_batches += 1

    # 8-bit batches (BFS)
    n_loops = rng.choice([4, 6, 8, 10, 12])
    step_fn = train_steps[n_loops]
    for batch in ds8.get_batch(split='train', batch_size=16, max_len=ml8):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        loss, grads = step_fn(model, opt_state, tk, pm)
        grads = grokfast(grads)
        updates, opt_state = optimizer.update(
            grads, opt_state, eqx.filter(model, eqx.is_array)
        )
        model = eqx.apply_updates(model, updates)
        epoch_loss += float(loss)
        n_batches += 1

    if epoch % 50 == 0:
        avg = epoch_loss / n_batches
        c8, t8 = quick_eval(model, 8, 8)
        print(f'Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit: {c8}/{t8} = {c8/t8:.1%}')

print('\nTraining complete.')
c8, t8 = quick_eval(model, 8, 8)
print(f'Final 8-bit: {c8}/{t8} = {c8/t8:.1%}')

In [ ]:
# Cell 5: Length generalization evaluation
print('=' * 50)
print('LENGTH GENERALIZATION EVALUATION')
print('NoPE + Input Injection + BFS Traces')
print('=' * 50)
print(f"\n{'Bits':>6} {'Loops':>6} {'Correct':>8} {'Total':>6} {'Accuracy':>10}")
print('-' * 44)

for n_bits, n_loops in [(4, 8), (8, 8), (16, 12), (32, 16)]:
    try:
        n_samples = 200 if n_bits <= 16 else 50
        c, t = quick_eval(model, n_bits, n_loops, n_samples=n_samples)
        print(f'{n_bits:6d} {n_loops:6d} {c:8d} {t:6d} {c/t:10.1%}')
    except Exception as e:
        print(f'{n_bits:6d} {n_loops:6d} {"ERROR":>8} {"":>6} {str(e)[:50]}')

print('\n' + '=' * 50)

In [ ]:
# Cell 6: Save model to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_nope_bfs.eqx'), model)
print(f'Model saved to {CKPT_DIR}')